# Session 会话管理教程 (v0.4)

本教程帮助你理解 Session 会话管理的概念和使用方式。

## 学习目标

1. 理解什么是 Session，与 Conversation 的区别
2. 掌握会话生命周期管理
3. 学会使用不同的存储后端
4. 理解上下文窗口管理

In [1]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", 
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

依赖安装完成！


In [2]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

项目根目录: d:\SVN\RogueRabbit
源码路径: d:\SVN\RogueRabbit\src


## 1. 什么是 Session？

Session 是**完整的会话对象**，包含：
- 会话元数据（ID、创建时间、状态等）
- 完整对话历史
- 系统提示词

### Session vs Conversation

| 特性 | Conversation | Session |
|------|-------------|---------|
| 类型 | 简单封装 | 完整管理 |
| 元数据 | 无 | 有（ID、时间、状态） |
| 生命周期 | 无 | 完整管理 |
| 持久化 | 不支持 | 支持 |
| 上下文管理 | 无 | 支持 |

In [3]:
# 导入 Session 相关类型
from rogue_rabbit.contracts import (
    Session, SessionMeta, SessionStatus,
    Message, Role
)

# 创建一个简单的 Session
session = Session(
    meta=SessionMeta(metadata={"user": "demo", "topic": "learning"}),
    system_prompt="你是一个有帮助的助手。"
)

print(f"Session ID: {session.meta.session_id}")
print(f"状态: {session.meta.status.value}")
print(f"创建时间: {session.meta.created_at}")
print(f"元数据: {session.meta.metadata}")

Session ID: d0c07065
状态: active
创建时间: 2026-03-26 21:13:36.147127
元数据: {'user': 'demo', 'topic': 'learning'}


In [4]:
# 添加消息到 Session
session.add_message(Message(role=Role.USER, content="你好"))
session.add_message(Message(role=Role.ASSISTANT, content="你好！有什么可以帮助你的？"))

print(f"消息数量: {len(session.messages)}")
print(f"最后更新: {session.meta.updated_at}")

print("\n对话历史:")
for msg in session.messages:
    print(f"  [{msg.role.value}]: {msg.content}")

消息数量: 2
最后更新: 2026-03-27 09:55:28.083994

对话历史:
  [user]: 你好
  [assistant]: 你好！有什么可以帮助你的？


## 2. 会话生命周期

Session 有三种状态：
- **ACTIVE**: 活跃状态，可以正常对话
- **IDLE**: 暂停状态，可以恢复
- **CLOSED**: 关闭状态，不可恢复

状态转换：`ACTIVE <-> IDLE -> CLOSED`

In [ ]:
# 使用 SessionManager 管理会话
from rogue_rabbit.core import SessionManager
from rogue_rabbit.runtime import MemorySessionStore
from rogue_rabbit.adapters import GLMClient

# 创建会话管理器
store = MemorySessionStore()
llm = GLMClient()
manager = SessionManager(store=store, llm_client=llm)

# 创建新会话
session = manager.create(
    system_prompt="你是一个有帮助的助手，回答要简洁。",
    metadata={"purpose": "demo"}
)

print(f"创建会话: {session.meta.session_id}")
print(f"初始状态: {session.meta.status.value}")

In [ ]:
# 进行对话
response = manager.chat(session.meta.session_id, "介绍一下你自己")
print(f"用户: 介绍一下你自己")
print(f"AI: {response[:100]}...")

# 暂停会话
manager.pause(session.meta.session_id)
session = manager.get(session.meta.session_id)
print(f"\n暂停后状态: {session.meta.status.value}")

# 恢复会话（通过 get 自动恢复）
response = manager.chat(session.meta.session_id, "继续")
print(f"\n恢复对话: {response[:50]}...")

In [ ]:
# 关闭会话
manager.close(session.meta.session_id)
session = manager.get(session.meta.session_id)
print(f"关闭后状态: {session.meta.status.value}")

# 尝试在关闭的会话中对话
try:
    manager.chat(session.meta.session_id, "还能聊吗？")
except ValueError as e:
    print(f"异常: {e}")

## 3. 存储后端

Session 支持多种存储后端：
- **MemorySessionStore**: 内存存储，适合测试
- **FileSessionStore**: 文件存储，适合单机部署

In [ ]:
# 文件存储示例
import tempfile
from rogue_rabbit.runtime import FileSessionStore

with tempfile.TemporaryDirectory() as tmpdir:
    store_path = Path(tmpdir)
    file_store = FileSessionStore(store_path)
    
    # 创建并保存会话
    session = Session(
        meta=SessionMeta(metadata={"demo": "file"}),
        system_prompt="你是助手"
    )
    file_store.save(session)
    
    # 查看文件
    files = list(store_path.glob("*.json"))
    print(f"创建文件: {[f.name for f in files]}")
    
    # 加载会话
    loaded = file_store.load(session.meta.session_id)
    print(f"加载会话: {loaded.meta.session_id}")
    print(f"系统提示词: {loaded.system_prompt}")

## 4. 上下文窗口管理

长对话需要管理上下文窗口，避免超出 LLM token 限制。

支持多种截断策略：
- **KEEP_LAST**: 保留最近 N 条消息
- **KEEP_FIRST_LAST**: 保留首尾消息
- **SUMMARIZE**: 生成摘要（需要 LLM 支持）

In [ ]:
from rogue_rabbit.core import (
    ContextWindowManager, 
    ContextWindowConfig, 
    TruncationStrategy
)

# 创建测试消息
messages = [Message(role=Role.USER, content=f"消息 {i}") for i in range(10)]

print(f"原始消息数: {len(messages)}")
print(f"内容: {[m.content for m in messages]}")

In [ ]:
# KEEP_LAST 策略
config = ContextWindowConfig(
    max_messages=5,
    strategy=TruncationStrategy.KEEP_LAST
)
manager = ContextWindowManager(config=config)
managed = manager.manage(messages)

print(f"KEEP_LAST 截断后: {len(managed)} 条")
print(f"内容: {[m.content for m in managed]}")

In [ ]:
# KEEP_FIRST_LAST 策略
config = ContextWindowConfig(
    max_messages=10,
    strategy=TruncationStrategy.KEEP_FIRST_LAST,
    keep_first=2,
    keep_last=3
)
manager = ContextWindowManager(config=config)
managed = manager.manage(messages)

print(f"KEEP_FIRST_LAST 截断后: {len(managed)} 条")
print(f"内容: {[m.content for m in managed]}")

In [ ]:
# 与 SessionManager 集成
store = MemorySessionStore()
llm = GLMClient()
context_manager = ContextWindowManager(
    config=ContextWindowConfig(max_messages=10)
)
session_manager = SessionManager(
    store=store,
    llm_client=llm,
    context_window_manager=context_manager
)

print("SessionManager 与 ContextWindowManager 集成完成")

## 5. 会话序列化

Session 支持序列化和反序列化，便于备份和迁移。

In [ ]:
import json

# 创建会话
session = Session(
    meta=SessionMeta(metadata={"version": "1.0"}),
    system_prompt="你是助手"
)
session.add_message(Message(role=Role.USER, content="你好"))
session.add_message(Message(role=Role.ASSISTANT, content="你好！"))

# 序列化为字典
data = session.to_dict()
print("序列化结果:")
print(json.dumps(data, ensure_ascii=False, indent=2)[:500] + "...")

In [ ]:
# 反序列化
restored = Session.from_dict(data)
print(f"恢复会话 ID: {restored.meta.session_id}")
print(f"消息数量: {len(restored.messages)}")
print(f"系统提示词: {restored.system_prompt}")

## 总结

### 核心概念

1. **Session**: 完整会话对象，包含元数据和历史
2. **SessionManager**: 会话生命周期管理
3. **SessionStore**: 存储抽象（内存/文件）
4. **ContextWindowManager**: 上下文窗口控制

### 会话状态

- `ACTIVE` <-> `IDLE` -> `CLOSED`

### 存储选择

- **Memory**: 测试、临时使用
- **File**: 单机部署、持久化

### 下一步

- 运行 `experiments/10_session_basic.py`
- 运行 `experiments/11_session_persistence.py`
- 尝试创建多会话应用